In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, '../Week-2-3-4')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,RobustScaler,TargetEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVR,NuSVR # regression
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from scipy.stats import pearsonr

In [ ]:
df_raw_20 = pd.read_csv("../Datasets/processed//uhpc_dataset/raw_dropped_features_20.csv")
df_raw_50 = pd.read_csv("../Datasets/processed//uhpc_dataset/raw_dropped_features_50.csv")
df_recoded_20 = pd.read_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features_20.csv")
df_recoded_50 = pd.read_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features_50.csv")

dfs = [df_raw_20, df_raw_50, df_recoded_20, df_recoded_50]
df_names = ['raw_20', 'raw_50', 'recoded_20', 'recoded_50']
models = [KNeighborsRegressor, NuSVR, SVR]
model_names_list = ['KNeighborsRegressor', 'NuSVR', 'SVR']


In [ ]:
for df, df_name in zip(dfs, df_names):
    for model_class, model_name in zip(models, model_names_list):

        target_col = 'cs_28d'
        X = df[df.columns[df.columns != target_col]]
        y = df[target_col]

        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.30, random_state=42
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.50, random_state=42
        )
        
        
        X = df.drop(columns=[target_col])

        str_cols = X.select_dtypes(include='object').columns
        one_hot_columns = str_cols[X[str_cols].nunique() <= 10].tolist()
        k_fold_columns = str_cols[X[str_cols].nunique() > 10].tolist()
        numerical_cols = X.select_dtypes(include='number').columns.tolist()
        
        
        numerical_pipeline = Pipeline([
        ('imputer', KNNImputer()),    #KNNImputer() or  SimpleImputer(strategy='median')
        ('scaler', RobustScaler())
        ])
        
        numerical_pipeline.fit(X_train[numerical_cols])
        
        preprocessor = ColumnTransformer([
        ('num', numerical_pipeline, numerical_cols),
        ('ohe', OneHotEncoder(), one_hot_columns),
        ('target', TargetEncoder(cv=5), k_fold_columns)
        ])
        
        
        full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_class())
        ])
        
        full_pipeline.fit(X_train, y_train)
        
        y_pred_train = full_pipeline.predict(X_train)
        
        rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
        
        y_pred = full_pipeline.predict(X_test)

        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        r, _ = pearsonr(y_test, y_pred)

        print(f"Dataset: {df_name:12} | Model: {model_name:20} | RMSE test: {rmse_test:.4f} | RMSE train: {rmse_train:.4f} | MAE: {mae:.4f} | R2: {r2:.4f} | R: {r:.4f}")

    
    

C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = X.select_dtypes(include='object').columns
C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

Dataset: raw_20       | Model: KNeighborsRegressor  | RMSE test: 22.8564 | RMSE train: 20.8523 | MAE: 16.6631 | R2: 0.6197 | R: 0.7933
Dataset: raw_20       | Model: NuSVR                | RMSE test: 34.2587 | RMSE train: 34.0744 | MAE: 26.4213 | R2: 0.1455 | R: 0.5183


C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = X.select_dtypes(include='object').columns


Dataset: raw_20       | Model: SVR                  | RMSE test: 33.9950 | RMSE train: 33.7819 | MAE: 25.9739 | R2: 0.1586 | R: 0.5340
Dataset: raw_50       | Model: KNeighborsRegressor  | RMSE test: 21.4620 | RMSE train: 19.2482 | MAE: 15.9465 | R2: 0.6646 | R: 0.8192


C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = X.select_dtypes(include='object').columns
C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

...

Dataset: recoded_20   | Model: SVR                  | RMSE test: 33.9821 | RMSE train: 33.7806 | MAE: 25.9047 | R2: 0.1593 | R: 0.5403
Dataset: recoded_50   | Model: KNeighborsRegressor  | RMSE test: 21.9446 | RMSE train: 19.0175 | MAE: 16.4860 | R2: 0.6494 | R: 0.8088


C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = X.select_dtypes(include='object').columns
C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

Dataset: recoded_50   | Model: NuSVR                | RMSE test: 34.2967 | RMSE train: 34.0900 | MAE: 26.3967 | R2: 0.1436 | R: 0.5190


C:\Users\shoai\AppData\Local\Temp\ipykernel_44624\4055664488.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = X.select_dtypes(include='object').columns


Dataset: recoded_50   | Model: SVR                  | RMSE test: 34.0066 | RMSE train: 33.7989 | MAE: 25.9163 | R2: 0.1580 | R: 0.5405
